1️⃣ Importamos librerías y cargamos el CSV de Data Science Job salaries

In [2]:
import pandas as pd
import numpy as np
# Configura Pandas para que use dos decimales y elimine la notación científica
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Cargar dataset
df = pd.read_csv("salaries.csv")

# Mostrar primeras filas
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'salaries.csv'

2️⃣ Dimensiones del dataset (.shape)

In [6]:
print("Número de filas y columnas:")
print(df.shape)

Número de filas y columnas:
(105434, 11)


3️⃣ Información general (.info())

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105434 entries, 0 to 105433
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   work_year           105434 non-null  int64 
 1   experience_level    105434 non-null  object
 2   employment_type     105434 non-null  object
 3   job_title           105434 non-null  object
 4   salary              105434 non-null  int64 
 5   salary_currency     105434 non-null  object
 6   salary_in_usd       105434 non-null  int64 
 7   employee_residence  105434 non-null  object
 8   remote_ratio        105434 non-null  int64 
 9   company_location    105434 non-null  object
 10  company_size        105434 non-null  object
dtypes: int64(4), object(7)
memory usage: 8.8+ MB


4️⃣ Tipos de datos (.dtypes)

In [8]:
print(df.dtypes)

work_year              int64
experience_level      object
employment_type       object
job_title             object
salary                 int64
salary_currency       object
salary_in_usd          int64
employee_residence    object
remote_ratio           int64
company_location      object
company_size          object
dtype: object


In [20]:
# 1. Filtrar por residencia en España
df_espana = df[df['employee_residence'] == 'ES']

# 2. Obtener dimensiones (filas, columnas)
filas, columnas = df_espana.shape

print(f"Registros (personas en España): {filas}")
print(f"Variables (columnas): {columnas}")

Registros (personas en España): 233
Variables (columnas): 11


5️⃣ Estadísticos descriptivos (.describe())

📊 ¿Qué significan las Columnas?

*   work_year (Año de trabajo): Indica el año en el que se registraron los datos (va del 2020 al 2025).

*   salary (Salario original): El salario bruto en la moneda local del país donde se paga (por eso hay números gigantescos como 30 millones, porque pueden ser yenes, rupias, etc.).

*   salary_in_usd (Salario en USD): Esta es la columna más importante para comparar. Todos los salarios convertidos a dólares estadounidenses.

*  remote_ratio (Ratio de trabajo remoto): El porcentaje de presencialidad/remoto. Típicamente 0 significa 100% presencial, 50 es híbrido y 100 es 100% remoto.

In [9]:
df.describe()

,work_year,salary,salary_in_usd,remote_ratio
count,105434.00,105434.00,105434.00,105434.00
mean,2024.19,162690.78,158018.51,21.10
std,0.67,213723.60,74401.71,40.71
min,2020.00,14000.00,15000.00,0.00
25%,2024.00,106400.00,106400.00,0.00
50%,2024.00,147100.00,147000.00,0.00
75%,2025.00,199700.00,199000.00,0.00
max,2025.00,30400000.00,800000.00,100.00


In [10]:
df.describe(include='all')

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
count,105434.00,105434,105434,105434,105434.00,105434,105434.00,105434,105434.00,105434,105434
unique,NaN,4,4,347,NaN,26,NaN,98,NaN,92,3
top,NaN,SE,FT,Data Scientist,NaN,USD,NaN,US,NaN,US,M
freq,NaN,61396,104865,14938,NaN,99906,NaN,94777,NaN,94834,102200
mean,2024.19,NaN,NaN,NaN,162690.78,NaN,158018.51,NaN,21.10,NaN,NaN
std,0.67,NaN,NaN,NaN,213723.60,NaN,74401.71,NaN,40.71,NaN,NaN
min,2020.00,NaN,NaN,NaN,14000.00,NaN,15000.00,NaN,0.00,NaN,NaN
25%,2024.00,NaN,NaN,NaN,106400.00,NaN,106400.00,NaN,0.00,NaN,NaN
50%,2024.00,NaN,NaN,NaN,147100.00,NaN,147000.00,NaN,0.00,NaN,NaN
75%,2025.00,NaN,NaN,NaN,199700.00,NaN,199000.00,NaN,0.00,NaN,NaN


In [11]:
estadisticos = pd.DataFrame({
    'Media': df.select_dtypes(include='number').mean(),
    'Mediana': df.select_dtypes(include='number').median(),
    'Desv_Estandar': df.select_dtypes(include='number').std(),
    'Minimo': df.select_dtypes(include='number').min(),
    'Maximo': df.select_dtypes(include='number').max()
})

estadisticos

,Media,Mediana,Desv_Estandar,Minimo,Maximo
work_year,2024.19,2024.00,0.67,2020,2025
salary,162690.78,147100.00,213723.60,14000,30400000
salary_in_usd,158018.51,147000.00,74401.71,15000,800000
remote_ratio,21.10,0.00,40.71,0,100


Comparar media y mediana , para detectar asimetrías y posibles outliers.

Interpretación:

Media ≈ Mediana → distribución bastante simétrica.
Media > Mediana → valores altos extremos (sesgo positivo).
Media < Mediana → valores bajos extremos (sesgo negativo).

In [12]:
comparacion = pd.DataFrame({
    'Media': df.select_dtypes(include='number').mean(),
    'Mediana': df.select_dtypes(include='number').median()
})

comparacion['Diferencia'] = comparacion['Media'] - comparacion['Mediana']

comparacion

,Media,Mediana,Diferencia
work_year,2024.19,2024.00,0.19
salary,162690.78,147100.00,15590.78
salary_in_usd,158018.51,147000.00,11018.51
remote_ratio,21.10,0.00,21.10


6️⃣ Separar variables numéricas y categóricas

In [13]:
variables_numericas = df.select_dtypes(include=np.number)

variables_categoricas = df.select_dtypes(include='object')

print("Variables numéricas:")
print(variables_numericas.columns.tolist())

print("\nVariables categóricas:")
print(variables_categoricas.columns.tolist())

Variables numéricas:
['work_year', 'salary', 'salary_in_usd', 'remote_ratio']

Variables categóricas:
['experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location', 'company_size']


Categoría frecuente de cada variable:

In [14]:
for col in df.select_dtypes(include='object').columns:
    print(f"\n--- {col} ---")
    print("Categoría más frecuente:")
    print(df[col].mode()[0])


--- experience_level ---
Categoría más frecuente:
SE

--- employment_type ---
Categoría más frecuente:
FT

--- job_title ---
Categoría más frecuente:
Data Scientist

--- salary_currency ---
Categoría más frecuente:
USD

--- employee_residence ---
Categoría más frecuente:
US

--- company_location ---
Categoría más frecuente:
US

--- company_size ---
Categoría más frecuente:
M


In [15]:
resumen_cat = pd.DataFrame({
    'Valores_Unicos': df.select_dtypes(include='object').nunique(),
    'Categoria_Mas_Frecuente': df.select_dtypes(include='object').mode().iloc[0],
    'Frecuencia': [
        df[col].value_counts().iloc[0]
        for col in df.select_dtypes(include='object').columns
    ]
})

resumen_cat

,Valores_Unicos,Categoria_Mas_Frecuente,Frecuencia
experience_level,4,SE,61396
employment_type,4,FT,104865
job_title,347,Data Scientist,14938
salary_currency,26,USD,99906
employee_residence,98,US,94777
company_location,92,US,94834
company_size,3,M,102200


7️⃣ Detectar valores nulos

In [16]:
nulos = pd.DataFrame({
    "Nulos": df.isnull().sum(),
    "Porcentaje (%)": round((df.isnull().sum()/len(df))*100,2)
})

nulos.sort_values("Porcentaje (%)", ascending=False)

,Nulos,Porcentaje (%)
work_year,0,0.00
experience_level,0,0.00
employment_type,0,0.00
job_title,0,0.00
salary,0,0.00
salary_currency,0,0.00
salary_in_usd,0,0.00
employee_residence,0,0.00
remote_ratio,0,0.00
company_location,0,0.00


8️⃣ Comprobar duplicados

In [17]:
duplicados = df.duplicated().sum()

print(f"Registros duplicados: {duplicados}")

Registros duplicados: 52997


9️⃣ Resumen automático del dataset

In [18]:
print("RESUMEN GENERAL")
print("-"*50)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

print(f"\nVariables numéricas: {len(variables_numericas.columns)}")
print(f"Variables categóricas: {len(variables_categoricas.columns)}")

print(f"\nValores nulos totales: {df.isnull().sum().sum()}")

print(f"Duplicados: {df.duplicated().sum()}")

RESUMEN GENERAL
--------------------------------------------------
Filas: 105434
Columnas: 11

Variables numéricas: 4
Variables categóricas: 7

Valores nulos totales: 0
Duplicados: 52997


Filtramos por los datos solo de España


In [19]:
# 1. Filtrar el dataset para personas que residen en España ('ES')
df_espana = df[df['employee_residence'] == 'ES']

# 2. Obtener el número de registros (datos de salarios) de España
num_salarios_espana = len(df_espana)

# 3. Mostrar los resultados
print(f"El número total de salarios registrados para personas en España es: {num_salarios_espana}")
print("\nPrimeras filas de los datos de España:")
df_espana.head()

El número total de salarios registrados para personas en España es: 233

Primeras filas de los datos de España:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
1352,2025,MI,FT,AI Engineer,46000,EUR,48421,ES,0,ES,M
1353,2025,MI,FT,AI Engineer,26000,EUR,27368,ES,0,ES,M
2700,2025,SE,FT,Data Engineer,40000,EUR,42105,ES,0,ES,M
2701,2025,SE,FT,Data Engineer,35000,EUR,36842,ES,0,ES,M
3138,2025,MI,FT,Backend Developer,32000,EUR,33684,ES,100,ES,M


# Observaciones iniciales

## Estructura del dataset

- El dataset contiene XXXX registros y XX variables.
- Se observan variables tanto numéricas como categóricas.
- La mayoría de variables describen características relacionadas con salarios y puestos de trabajo.

## Calidad de los datos

- Existen valores nulos en algunas columnas.
- Será necesario evaluar si los nulos deben eliminarse o imputarse.
- Se han detectado XX registros duplicados.

## Variables relevantes

- La variable salary parece especialmente importante para el análisis.
- También destacan las variables relacionadas con experiencia, ubicación y tipo de empleo.

## Próximos pasos

1. Analizar valores nulos.
2. Revisar duplicados.
3. Detectar valores atípicos.
4. Preparar los datos para el análisis estadístico y las visualizaciones.